# tAilorCode — envío autocontenido

Ejecuta este notebook completo para crear `/kaggle/working/submission.zip`. Los archivos del agente están integrados. No necesita GitHub, el dataset privado de tAilorCode, Docker ni instalar librerías. Conserva los datos de la competición asociados en Kaggle.

El ZIP contiene la configuración del agente; la evaluación y puntuación corresponden al sistema de Kaggle.

In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import zipfile

# Self-contained: no network, external input files, GPU or third-party packages.
embedded_files = {'agent.yaml': 'name: tAilorCode\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: A grounded autonomous coding agent for verified Python repository fixes.\ninstruction: |\n  You are tAilorCode, an autonomous software engineering agent working inside /workspace.\n\n  Resolve the supplied Python repository issue with the smallest source-code patch that passes verification tests.\n\n  Workflow:\n  1. Read the issue and identify acceptance criteria.\n  2. Inspect the repository and relevant source and test files.\n  3. Form an evidence-based root-cause hypothesis before editing.\n  4. Use code-intelligence tools with concrete symbol names when available.\n  5. Make a minimal source-only change. Never modify tests or test configuration.\n  6. Run targeted tests or focused assertions. Do not run a bare full-repository test command.\n  7. Remove temporary files and review the final diff.\n  8. Call submit_patch() only after verification. It must be your final tool call.\n\n  The environment is offline. Do not install packages or download files. Work only under /workspace and optimize for a correct, reviewable, reproducible patch.\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\n  - agent_tool:\n      config_path: sub_agents/code_analyzer.yaml\n      skip_summarization: true\ngenerate_content_config:\n  temperature: 0.2\n  top_p: 0.95\n  max_output_tokens: 16384\n  thinking_config:\n    thinking_budget: 4096\n    include_thoughts: true\n', 'eval_config.yaml': 'evaluation:\n  timeout_seconds: 300\n  max_tool_calls: 50\n  max_time_minutes: 30\n  max_turns: 100\n', 'sub_agents/code_analyzer.yaml': "name: tAilorCode_analyzer\ndescription: A read-only specialist that locates relevant files, symbols, and root causes.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: |\n  You are tAilorCode's read-only repository analyst.\n\n  Use read_file, search_similar_code, get_code_neighbors, and get_code_subgraph to locate the likely root cause. Do not modify files or run commands.\n\n  Return a concise report with relevant paths and symbols, evidence, root cause, and the smallest implementation direction for the main agent.\ntools:\n  - read_file\n  - search_similar_code\n  - get_code_neighbors\n  - get_code_subgraph\ngenerate_content_config:\n  temperature: 0.2\n  top_p: 0.95\n  max_output_tokens: 16384\n  thinking_config:\n    thinking_budget: 4096\n    include_thoughts: true\n"}

working_root = Path('/kaggle/working') if Path('/kaggle').is_dir() else Path.cwd()
working_root.mkdir(parents=True, exist_ok=True)
archive_path = working_root / 'submission.zip'

for name, content in embedded_files.items():
    path = PurePosixPath(name)
    assert not path.is_absolute() and '..' not in path.parts
    assert '!include' not in content
assert 'model: gemma-4-31b-it-qat-w4a16-ct' in embedded_files['agent.yaml']
assert 'model: gemma-4-31b-it-qat-w4a16-ct' in embedded_files['sub_agents/code_analyzer.yaml']

with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name, content in embedded_files.items():
        archive.writestr(name, content.encode('utf-8'))

with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(embedded_files)
    assert 'agent.yaml' in archive.namelist()
    for name, content in embedded_files.items():
        assert archive.read(name).decode('utf-8') == content
    assert sum(info.file_size for info in archive.infolist()) < 3 * 1024**3
    print('Contenido:', ', '.join(archive.namelist()))

print('ZIP creado y verificado:', archive_path)
print('Bytes:', archive_path.stat().st_size)
print('SHA256:', hashlib.sha256(archive_path.read_bytes()).hexdigest())
print('La creación del ZIP no ejecuta el agente ni confirma una puntuación en Kaggle.')


## Enviar en Kaggle

1. Importa este `.ipynb` en el editor del notebook de la competición.
2. Pulsa **Save Version → Save & Run All** y espera a que termine correctamente.
3. Comprueba que la salida de esa versión contiene `submission.zip`.
4. Selecciona esa versión para enviar a la competición y elige `submission.zip` si Kaggle solicita el archivo.
5. Si falla, copia el mensaje exacto y los logs de esa versión. Un ZIP generado no garantiza que el agente supere la evaluación.